# 1. Load Data from all the stock csv files in directory

In [1]:
import datetime
import glob
import numpy as np
import pandas as pd

# Get CSV files list from a folder
csv_files = glob.glob("./TSLA-Stocks/TSLA16*.csv")

# Read each CSV file into DataFrame
# This creates a list of dataframes
df_list = (pd.read_csv(file) for file in csv_files)

# Concatenate all DataFrames
df   = pd.concat(df_list, ignore_index=True)

## 1.1 Drop duplicates & check for missing values

In [2]:
# Drop duplicate entries
df.drop_duplicates(subset=['date'], keep='first', inplace=True)

In [3]:
# verify there are no duplicate values
df["date"].is_unique

True

## 1.2 Print out all dates with confirming # of quotes (23400) and non-conforming number of quotes

In [4]:
df['date2_str']= df['date'][::].str.slice(stop=10)
pd_group_cnt = df.groupby(['date2_str'])['date2_str'].count().to_frame()
print("Confirming / correct number of quotes: 23,400")
print(pd_group_cnt.loc[pd_group_cnt['date2_str']==23400] )
pd.set_option('display.max_rows', None)
print("Dates Missing quotes")
print(pd_group_cnt.loc[pd_group_cnt['date2_str']!=23400] )
pd.set_option('display.max_rows', 10)

Confirming / correct number of quotes: 23,400
            date2_str
date2_str            
2022-03-04      23400
2022-03-07      23400
2022-03-08      23400
2022-03-09      23400
2022-03-10      23400
...               ...
2022-10-03      23400
2022-10-04      23400
2022-10-05      23400
2022-10-06      23400
2022-10-07      23400

[151 rows x 1 columns]
Dates Missing quotes
            date2_str
date2_str            
2022-03-03       5406


In [5]:
print()

# 2.0 Describe data

In [6]:
df = df.sort_index(ascending=True)
#df = df.tail(10000)
#df = df.tail(30000)
df.describe()

,open,high,low,close,volume,average,barCount
count,3.538806e+06,3.538806e+06,3.538806e+06,3.538806e+06,3.538806e+06,3.538806e+06,3.538806e+06
mean,2.792378e+02,2.792584e+02,2.792166e+02,2.792375e+02,1.981523e+01,2.792374e+02,4.281384e+00
std,3.920246e+01,3.920186e+01,3.920304e+01,3.920243e+01,5.397667e+01,3.920246e+01,7.603019e+00
min,2.068667e+02,2.068733e+02,2.068567e+02,2.068733e+02,0.000000e+00,2.068677e+02,0.000000e+00
25%,2.444167e+02,2.444300e+02,2.443967e+02,2.444133e+02,1.000000e+00,2.444140e+02,1.000000e+00
50%,2.790100e+02,2.790300e+02,2.789967e+02,2.790100e+02,7.500000e+00,2.790095e+02,2.000000e+00
75%,3.031067e+02,3.031300e+02,3.030867e+02,3.031067e+02,2.331000e+01,3.031067e+02,5.000000e+00
max,3.841867e+02,3.842900e+02,3.840133e+02,3.841867e+02,3.025596e+04,3.842223e+02,3.700000e+02


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 3538806 entries, 0 to 3612599
Data columns (total 9 columns):
 #   Column     Dtype  
---  ------     -----  
 0   date       object 
 1   open       float64
 2   high       float64
 3   low        float64
 4   close      float64
 5   volume     float64
 6   average    float64
 7   barCount   int64  
 8   date2_str  object 
dtypes: float64(6), int64(1), object(2)
memory usage: 270.0+ MB


# 3.0 Add computed columns

In [8]:
df.set_index('date')
df = df.sort_index()

In [9]:
# need this column to compute average of averages
df['_weighted_vol_avg'] = df['volume'] * df['average']

In [10]:
# lambda functions

#return 1st value in series
def firstValue(rows):
    return rows.iloc[0]

#return last value in series
def lastValue(rows):
    return rows.iloc[-1]

#return arrow indicator for boxed in values;
#   -1 below lower bound
#    0 inside the box
#   +1 above the max value
def arrow(new_amt, old_amt, box):
    if old_amt == np.nan:
        return np.nan
    if new_amt == np.nan:
        return np.nan
    if (new_amt - old_amt) <= (box * -1):
        return '-1'
    if (new_amt - old_amt) >= box:
        return '1'
    else:
        return '0'

#  lambda function to adds up the last 5 values, excluding the very last value
def sum_last_5(rows):
    #print ("[" , rows[-6:-1], rows[-6:-1].sum(), "]")
    return rows[-6:-1].sum()

#  lambda function returns lowest of the last 5 values, excluding the very last value
def min_last_5(rows):
    return rows.iloc[-6:-1].min()

#  lambda function returns higest of  the last 5 values, excluding the very last value
def max_last_5(rows):
    return rows.iloc[-6:-1].max()

# Store/Save 1 second windows for h1, h2, h3, h4, h5

In [ ]:
df['h1s_high_max'] = df['high'].rolling(window=2).agg( {'maxLast': lastValue})
df['h1s_low_min'] = df['low'].rolling(window=2).agg( {'minLast': lastValue})
df['h1s_barCount_sum'] = df['barCount'].rolling(window=2).agg( {'sumLast': lastValue})
df['h1s_volume_sum'] = df['volume'].rolling(window=2).agg( {'sumLast': lastValue})
df['h1s_average_avg'] = df['average'].rolling(window=2).agg( {'sumLast': lastValue})

In [ ]:
df['h2s_high_max'] = df['high'].rolling(window=3).agg( {'maxLast': lastValue})
df['h2s_low_min'] = df['low'].rolling(window=3).agg( {'minLast': lastValue})
df['h2s_barCount_sum'] = df['barCount'].rolling(window=3).agg( {'sumLast': lastValue})
df['h2s_volume_sum'] = df['volume'].rolling(window=3).agg( {'sumLast': lastValue})
df['h2s_average_avg'] = df['average'].rolling(window=3).agg( {'sumLast': lastValue})


In [ ]:
df['h3s_high_max'] = df['high'].rolling(window=4).agg( {'maxLast': lastValue})
df['h3s_low_min'] = df['low'].rolling(window=4).agg( {'minLast': lastValue})
df['h3s_barCount_sum'] = df['barCount'].rolling(window=4).agg( {'sumLast': lastValue})
df['h3s_volume_sum'] = df['volume'].rolling(window=4).agg( {'sumLast': lastValue})
df['h3s_average_avg'] = df['average'].rolling(window=4).agg( {'sumLast': lastValue})


In [ ]:
df['h4s_high_max'] = df['high'].rolling(window=5).agg( {'maxLast': lastValue})
df['h4s_low_min'] = df['low'].rolling(window=5).agg( {'minLast': lastValue})
df['h4s_barCount_sum'] = df['barCount'].rolling(window=5).agg( {'sumLast': lastValue})
df['h4s_volume_sum'] = df['volume'].rolling(window=5).agg( {'sumLast': lastValue})
df['h4s_average_avg'] = df['average'].rolling(window=5).agg( {'sumLast': lastValue})

## Compute 5 second window summary

In [ ]:
df['h5s_high_max'] = df['high'].rolling(window=6).agg( {'maxLast5': max_last_5})
df['h5s_low_min'] = df['low'].rolling(window=6).agg( {'minLast5': min_last_5})
df['h5s_barCount_sum'] = df['barCount'].rolling(window=6).agg( {'sumLast5': sum_last_5})
df['h5s_volume_sum'] = df['volume'].rolling(window=6).agg( {'sumLast5': sum_last_5})
df['_h5s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=6).agg({'SumLast5': sum_last_5})

df['h5s_average_avg'] = df['_h5s_weighted_vol_avg_sum'] / df['h5s_volume_sum']
df.drop(columns=['_h5s_weighted_vol_avg_sum'])
#

## Compute 10 second window summary

In [ ]:
df['h10s_high_max'] = df['high'].rolling(window=11).agg( {'maxLast5': max_last_5})
df['h10s_low_min'] = df['low'].rolling(window=11).agg( {'minLast5': min_last_5})
df['h10s_barCount_sum'] = df['barCount'].rolling(window=11).agg( {'sumLast5': sum_last_5})
df['h10s_volume_sum'] = df['volume'].rolling(window=11).agg( {'sumLast5': sum_last_5})
df['_h10s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=11).agg({'SumLast5': sum_last_5})

df['h10s_average_avg'] = df['_h10s_weighted_vol_avg_sum'] / df['h10s_volume_sum']
df.drop(columns=['_h10s_weighted_vol_avg_sum'])

## Compute 15 second window summary

In [ ]:
df['h15s_high_max'] = df['high'].rolling(window=16).agg( {'maxLast5': max_last_5})
df['h15s_low_min'] = df['low'].rolling(window=16).agg( {'minLast5': min_last_5})
df['h15s_barCount_sum'] = df['barCount'].rolling(window=16).agg( {'sumLast5': sum_last_5})
df['h15s_volume_sum'] = df['volume'].rolling(window=16).agg( {'sumLast5': sum_last_5})
df['_h15s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=16).agg({'SumLast5': sum_last_5})

df['h15s_average_avg'] = df['_h15s_weighted_vol_avg_sum'] / df['h15s_volume_sum']
df.drop(columns=['_h15s_weighted_vol_avg_sum'])

In [ ]:
df = df.drop(columns=['_weighted_vol_avg'])

In [ ]:
df.set_index('date')
df = df.sort_index(ascending=False)
df.reset_index()
df.head(100)

## Compute Future 5 second window summary

In [ ]:
df['f5s_average'] = df['average'].rolling(window=7, closed='neither', min_periods=5).agg( {'last': lastValue})
df.head(100)

In [ ]:
pd.set_option('display.max_rows', 100)
df2 = df[::-1].copy()
df2['f5s_average'] = df2['average'].rolling(window=6).agg( {'firstValue': firstValue})
df2['f5s_10c_arrow'] = df2.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.10), axis=1)
df2['f5s_15c_arrow'] = df2.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.15), axis=1)
df2['f5s_20c_arrow'] = df2.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.20), axis=1)
df2['f5s_25c_arrow'] = df2.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.25), axis=1)
df2['f5s_30c_arrow'] = df2.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.30), axis=1)
df = df2[::-1].copy()

In [ ]:
df.to_csv("./TSLA-Stocks/ALL.csv", index=False)